# 01 · Build the dataset (Colab, no GPU needed)

**Runtime:** Runtime → Change runtime type → **CPU**. Nothing here needs a GPU, so it uses very few compute units.

**One-time setup:** in Google Drive, create a folder `patent-lora` in My Drive and upload
`patent-drawing-lora.zip` into it. The first cell unpacks it to `MyDrive/patent-lora/repo`.

Steps in this notebook:
1. Download ~3,000 US design-patent drawing sheets (DeepPatent).
2. **Pick ~25** in the thumbnail picker (saved to Drive as you tick).
3. Clean them: keep the main figure, strip labels, pad to 1024×1024.
4. **Review** and remove bad ones.
5. Draft captions with BLIP, then **fix them** in the caption editor.

Everything you pick or type is saved to Drive, so if Colab disconnects just re-run the cells.
If the downloaded sheets are gone after a restart, re-run step 1; your picks are kept.

In [ ]:
# --- Mount Drive and open the project (lives in MyDrive/patent-lora/repo) ----
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, zipfile
DRIVE_ROOT = "/content/drive/MyDrive/patent-lora"   # project, checkpoints and results all live here
REPO_DIR   = f"{DRIVE_ROOT}/repo"
ZIP_PATH   = f"{DRIVE_ROOT}/patent-drawing-lora.zip"  # upload the project zip here once
os.makedirs(DRIVE_ROOT, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError("Upload patent-drawing-lora.zip to MyDrive/patent-lora/ in Google Drive, then re-run.")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DRIVE_ROOT)
    shutil.move(f"{DRIVE_ROOT}/patent-drawing-lora", REPO_DIR)
    print("Project unpacked to", REPO_DIR)
%cd {REPO_DIR}

In [ ]:
# Widgets need this in Colab; the rest of the libraries are preinstalled.
from google.colab import output
output.enable_custom_widget_manager()
import sys
sys.path.insert(0, "scripts")
from pathlib import Path
RAW = Path("/content/candidates/raw")          # fast local disk; re-downloaded after a restart
SELECTION = Path("dataset/selection.txt")

## 1 · Download drawing sheets
Three shards spread across the dataset, about 1,000 sheets each. Takes a few minutes.

In [ ]:
if not any(RAW.rglob("*.png")):
    !python scripts/fetch_candidates.py --shards 0 120 240 --out /content/candidates
print(sum(1 for _ in RAW.rglob("*.png")), "sheets ready")

## 2 · Pick about 25 drawings
Tick sheets with **one clear object**. Aim for variety: tools, electronics, furniture, kitchenware,
wearables, vehicles (at most 2 per type), and a mix of views and shading styles.
Skip sheets that are mostly text or show many small figures. Ticks save to `dataset/selection.txt` instantly.

In [ ]:
from colab_ui import picker
picker(RAW, SELECTION)

## 3 · Clean the picks
Keeps the largest figure, removes labels and sheet numbers, pads to a white 1024×1024 square.

In [ ]:
!python scripts/prepare_data.py --raw /content/candidates/raw

## 4 · Review
Tick **remove** on any image with leftover labels, cut-off parts, two objects, or a messy scan, then click Apply.
Keep at least 20. If you drop below 20, go back to the picker, tick a few more, and re-run step 3.

In [ ]:
from colab_ui import review
review(Path("dataset/images"), SELECTION, Path("dataset/manifest.csv"))

## 5 · Rights check
Open a few `source_url` links from the manifest. US patent drawings are generally public domain,
but skip any patent whose text carries a copyright notice.

In [ ]:
import pandas as pd
pd.read_csv("dataset/manifest.csv")[["file_name", "patent", "issue_date", "source_url", "review"]]

## 6 · Draft captions with BLIP
About 1-2 minutes on CPU for 20 images. Style words are stripped automatically.

In [ ]:
!python scripts/caption.py draft

## 7 · Fix the captions
For every image: type **what the object is** (e.g. *a cordless power drill*) and choose the **view**.
Describe only the object and view. Never add style words like *line art* or *black and white*:
the trigger word `ptnt_style` must carry the style. Click **Save captions** when done.

In [ ]:
from colab_ui import caption_editor
caption_editor(Path("dataset/images"), Path("dataset/captions.csv"))

## Done
Check the final caption file below, then open **02_train_colab.ipynb** with an L4 GPU runtime.

In [ ]:
print(open("dataset/images/metadata.jsonl").read())